=============================================================================
Trip Purpose - Detailed
=============================================================================
Wkg questions:
1. How do trip purposes (HBW, HBO, NHB) vary by time of day and corridor?
2. What percentage of trips outside transit hours are work-related vs other purposes?
3. Which corridors show the highest demand for each trip purpose during off-hours?
=============================================================================

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set up formatting
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
comma_fmt = FuncFormatter(lambda x, p: format(int(x), ','))

=============================================================================
CONFIGURATION - EDIT THESE VALUES
=============================================================================

In [ ]:
# File paths
INPUT_FILE = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2013907_BI_Corridor_Volume_2025\2013907_BI_Corridor_Volume_2024_mf_traveler_trip_purpose_all.csv'
OUTPUT_DIR = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\output_trip_purpose'

In [ ]:
# Create output subdirectories
os.makedirs(OUTPUT_DIR, exist_ok=True)
for subdir in ['heatmaps', 'summary_tables', 'analysis_reports', 'purpose_breakdowns']:
    os.makedirs(os.path.join(OUTPUT_DIR, subdir), exist_ok=True)

=============================================================================
TRANSIT OPERATING HOURS -  
=============================================================================

In [ ]:
TRANSIT_CONFIG = {
    'monday':    {'start_hour': 4.5, 'end_hour': 19.25},
    'tuesday':   {'start_hour': 4.5, 'end_hour': 19.25},
    'wednesday': {'start_hour': 4.5, 'end_hour': 19.25},
    'thursday':  {'start_hour': 4.5, 'end_hour': 19.25},
    'friday':    {'start_hour': 4.5, 'end_hour': 19.25},
    'saturday':  {'start_hour': 9.0, 'end_hour': 18.0},
    'sunday':    {'start_hour': 9.0, 'end_hour': 16.0}
}

In [ ]:
# Trip purposes to analyze
TRIP_PURPOSES = {
    'Home to Work': 'Home to Work',
    'Home to Other': 'Home to Other',
    'Non-Home Based Trip': 'Non-Home Based'
}

In [ ]:
# Commercial corridors (middle filter zones)
COMMERCIAL_CORRIDORS = [
    'HIGH_SCHOOL_RD',
    'SPORTSMAN_CLUB',
    'SR305_MID',
    'SR305_N',
    'SR305_S',
    'WINSLOW_WAY'
]

In [ ]:
# Origin and destination zones (change if needed)
ORIGIN_ZONE = 'kitsap_broadly2'
DESTINATION_ZONE = 'kitsap_broadly4'

In [ ]:
# Volume column
VOLUME_COL = 'Average Daily O-M-D Traffic (StL Volume)'

In [ ]:
# Time periods for summary tables
TIME_PERIODS = {
    str(h): {
        'hours': [h],
        'display': f'{h % 12 or 12}{"am" if h < 12 else "pm"}'
    }
    for h in range(24)
}

=============================================================================
HELPER FUNCTIONS
=============================================================================

In [ ]:
def extract_hour(period_str):
    """Extract hour number from period string like '08: 7am-8am'"""
    try:
        return int(str(period_str).split(':')[0])
    except:
        return None

In [ ]:
def get_day_category(day_type):
    day_str = str(day_type).lower()
    if 'monday'    in day_str: return 'monday'
    if 'tuesday'   in day_str: return 'tuesday'
    if 'wednesday' in day_str: return 'wednesday'
    if 'thursday'  in day_str: return 'thursday'
    if 'friday'    in day_str: return 'friday'
    if 'saturday'  in day_str: return 'saturday'
    if 'sunday'    in day_str: return 'sunday'
    return 'other'

In [ ]:
def get_transit_status(hour, day_category):
    """Determine if a given hour is during transit service"""
    if pd.isna(hour) or day_category not in TRANSIT_CONFIG:
        return 'Unknown'
    
    config = TRANSIT_CONFIG[day_category]
    
    if hour >= config['start_hour'] and hour < config['end_hour']:
        return 'During Service'
    elif hour < config['start_hour']:
        return 'Before Service'
    else:
        return 'After Service'

In [ ]:
def calculate_trip_volumes(df):
    """Calculate trip volumes from percentages"""
    df = df.copy()
    for purpose in TRIP_PURPOSES.keys():
        if purpose in df.columns:
            df[purpose] = (df[VOLUME_COL] * df[purpose]).round(0)
    return df

=============================================================================
READ & PREP DATA
=============================================================================

In [ ]:
print("=" * 80)
print("98110 MOBILITY DEMAND ANALYSIS BY TRIP PURPOSE")
print("=" * 80)
print(f"\nAnalysis started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
try:
    print(f"\nReading data from: {INPUT_FILE}")
    df = pd.read_csv(INPUT_FILE)
    print(f"  Successfully loaded {len(df):,} records")
except Exception as e:
    print(f"ERROR: Could not read input file: {e}")
    print("Please check the file path and try again.")
    exit(1)

In [ ]:
# Calculate trip volumes
print("\nCalculating trip volumes from percentages...")
df = calculate_trip_volumes(df)

In [ ]:
# Add hour column
df['Hour'] = df['Day Part'].apply(extract_hour)
print(f"  Hours range: {df['Hour'].min()} to {df['Hour'].max()}")

In [ ]:
# Add day category
df['Day_Category'] = df['Day Type'].apply(get_day_category)

In [ ]:
# Add transit status
df['Transit_Status'] = df.apply(
    lambda row: get_transit_status(row['Hour'], row['Day_Category']), 
    axis=1
)

In [ ]:
# Add time period
def get_time_period(hour):
    if pd.isna(hour):
        return 'Daily Total'
    for period_name, period_info in TIME_PERIODS.items():
        if hour in period_info['hours']:
            return period_name
    return 'Other'

In [ ]:
df['Time_Period'] = df['Hour'].apply(get_time_period)

In [ ]:
# Separate hourly vs daily data
df_hourly = df[df['Day Part'] != '00: All Day (12am-12am)'].copy()
df_daily = df[df['Day Part'] == '00: All Day (12am-12am)'].copy()

In [ ]:
# Filter out aggregate "All Days"
df_hourly_individual = df_hourly[~df_hourly['Day Type'].astype(str).str.contains('All Days', na=False)].copy()
df_daily_individual = df_daily[~df_daily['Day Type'].astype(str).str.contains('All Days', na=False)].copy()

In [ ]:
print(f"\nData Overview:")
print(f"  Hourly records: {len(df_hourly_individual):,}")
print(f"  Daily total records: {len(df_daily_individual):,}")
print(f"  Day types: {sorted(df_hourly_individual['Day Type'].unique())}")
print(f"  Transit status distribution:")
print(df_hourly_individual['Transit_Status'].value_counts().to_string())

=============================================================================
FUNCTION: Summary tables by trip purpose
=============================================================================

In [ ]:
def create_purpose_summary(data_df, period_name, period_info, purpose):
    """
    Create summary table for a specific trip purpose and time period
    """
    # Filter for time period
    if period_name != 'All Day':
        period_data = data_df[data_df['Hour'].isin(period_info['hours'])]
    else:
        period_data = data_df
    
    if len(period_data) == 0:
        return None
    
    # Create pivot table
    summary = period_data.pivot_table(
        values=purpose,
        index='Middle Filter Zone Name',
        columns='Day Type',
        aggfunc='sum',
        fill_value=0
    )
    
    # Add totals
    summary.loc['TOTAL'] = summary.sum()
    summary['TOTAL'] = summary.sum(axis=1)
    
    return summary

=============================================================================
CREATE SUMMARY TABLES PER PURPOSE
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("CREATING TRIP PURPOSE SUMMARY TABLES")
print("=" * 80)

In [ ]:
purpose_summaries = {}

In [ ]:
for purpose_name, purpose_label in TRIP_PURPOSES.items():
    print(f"\n{purpose_label}:")
    purpose_summaries[purpose_name] = {}
    
    for period_name, period_info in TIME_PERIODS.items():
        summary = create_purpose_summary(
            df_hourly_individual, period_name, period_info, purpose_name
        )
        
        if summary is not None:
            purpose_summaries[purpose_name][period_name] = summary
            
            # Save to CSV
            filename = f'{purpose_name}_{period_name}_summary.csv'
            filepath = os.path.join(OUTPUT_DIR, 'summary_tables', filename)
            summary.to_csv(filepath)
            
            print(f"  {period_info['display']}: {len(summary)} rows")

=============================================================================
HEATMAPS: Hourly patterns by purpose for each corridor
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("CREATING TRIP PURPOSE HEATMAPS")
print("=" * 80)

In [ ]:
for corridor in COMMERCIAL_CORRIDORS:
    print(f"\n{corridor}:")
    
    # Filter data for this corridor
    corridor_data = df_hourly_individual[
        df_hourly_individual['Middle Filter Zone Name'] == corridor
    ]
    
    if len(corridor_data) == 0:
        print(f"  No data found")
        continue
    
    # Create a figure with subplots for each purpose
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for idx, (purpose_name, purpose_label) in enumerate(TRIP_PURPOSES.items()):
        ax = axes[idx]
        
        # Create pivot table
        pivot_data = corridor_data.pivot_table(
            values=purpose_name,
            index='Hour',
            columns='Day Type',
            aggfunc='sum',
            fill_value=0
        )
        
        if len(pivot_data) == 0:
            ax.text(0.5, 0.5, 'No Data', ha='center', va='center')
            continue
        
        # Dynamic color scaling
        max_val = pivot_data.max().max()
        
        # Create heatmap
        sns.heatmap(pivot_data, 
                    annot=True, 
                    fmt='.0f',
                    cmap='YlOrRd',
                    linewidths=0.5,
                    vmin=0,
                    vmax=max_val,
                    ax=ax,
                    cbar_kws={'label': 'Trip Volume'})
        
        ax.set_title(f'{purpose_label}\n(Max: {max_val:,.0f})')
        ax.set_xlabel('Day Type')
        ax.set_ylabel('Hour of Day')
    
    plt.suptitle(f'Trip Purpose Analysis - {corridor}', fontsize=14, y=1.02)
    plt.tight_layout()
    
    # Save
    filename = f'Heatmap_{corridor}_purposes.pdf'
    filepath = os.path.join(OUTPUT_DIR, 'heatmaps', filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Saved heatmap")

=============================================================================
PURPOSE PROPORTIONS
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("TRIP PURPOSE PROPORTION ANALYSIS")
print("=" * 80)

In [ ]:
# Calculate purpose proportions by time period and transit status
purpose_proportions = []

In [ ]:
for period_name, period_info in TIME_PERIODS.items():
    period_data = df_hourly_individual[
        df_hourly_individual['Hour'].isin(period_info['hours'])
    ]
    
    for transit_status in ['During Service', 'Before Service', 'After Service']:
        status_data = period_data[period_data['Transit_Status'] == transit_status]
        
        if len(status_data) == 0:
            continue
        
        total_trips = status_data[list(TRIP_PURPOSES.keys())].sum().sum()
        
        if total_trips > 0:
            for purpose_name, purpose_label in TRIP_PURPOSES.items():
                purpose_total = status_data[purpose_name].sum()
                proportion = (purpose_total / total_trips * 100) if total_trips > 0 else 0
                
                purpose_proportions.append({
                    'Time_Period': period_info['display'],
                    'Transit_Status': transit_status,
                    'Trip_Purpose': purpose_label,
                    'Volume': purpose_total,
                    'Proportion': proportion
                })

In [ ]:
purpose_prop_df = pd.DataFrame(purpose_proportions)
print("\nTrip Purpose Proportions by Time Period and Transit Status:")
print(purpose_prop_df.to_string(index=False))

=============================================================================
CORRIDOR-SPECIFIC PURPOSES
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("CORRIDOR-SPECIFIC PURPOSE ANALYSIS")
print("=" * 80)

In [ ]:
corridor_purpose_analysis = []

In [ ]:
for corridor in COMMERCIAL_CORRIDORS:
    corridor_data = df_hourly_individual[
        df_hourly_individual['Middle Filter Zone Name'] == corridor
    ]
    
    if len(corridor_data) == 0:
        continue
    
    for transit_status in ['During Service', 'Before Service', 'After Service']:
        status_data = corridor_data[corridor_data['Transit_Status'] == transit_status]
        
        if len(status_data) == 0:
            continue
        
        row = {'Corridor': corridor, 'Transit_Status': transit_status}
        total = 0
        
        for purpose_name, purpose_label in TRIP_PURPOSES.items():
            volume = status_data[purpose_name].sum()
            row[purpose_label] = volume
            total += volume
        
        if total > 0:
            for purpose_name, purpose_label in TRIP_PURPOSES.items():
                row[f'{purpose_label}_Pct'] = (row[purpose_label] / total * 100)
            
            corridor_purpose_analysis.append(row)

In [ ]:
corridor_purpose_df = pd.DataFrame(corridor_purpose_analysis)
print("\nCorridor Analysis by Transit Status:")
print(corridor_purpose_df.to_string())

=============================================================================
PEAK HOUR BY PURPOSE
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("PEAK HOUR ANALYSIS BY PURPOSE")
print("=" * 80)

In [ ]:
peak_hours_by_purpose = []

In [ ]:
for purpose_name, purpose_label in TRIP_PURPOSES.items():
    for corridor in COMMERCIAL_CORRIDORS:
        corridor_data = df_hourly_individual[
            df_hourly_individual['Middle Filter Zone Name'] == corridor
        ]
        
        if len(corridor_data) == 0:
            continue
        
        # Find peak hour for this purpose
        peak_idx = corridor_data[purpose_name].idxmax()
        peak_row = corridor_data.loc[peak_idx]
        
        peak_hours_by_purpose.append({
            'Trip_Purpose': purpose_label,
            'Corridor': corridor,
            'Peak_Hour': f"{int(peak_row['Hour']):02d}:00-{int(peak_row['Hour'])+1:02d}:00",
            'Peak_Volume': peak_row[purpose_name],
            'Day_Type': peak_row['Day Type'],
            'Transit_Status': peak_row['Transit_Status']
        })

In [ ]:
peak_purpose_df = pd.DataFrame(peak_hours_by_purpose)
print("\nPeak Hours by Purpose:")
print(peak_purpose_df.to_string(index=False))

=============================================================================
EXPORT ALL RESULTS
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("EXPORTING RESULTS")
print("=" * 80)

In [ ]:
# Create Excel file with multiple sheets
excel_path = os.path.join(OUTPUT_DIR, 'complete_trip_purpose_analysis.xlsx')
with pd.ExcelWriter(excel_path) as writer:
    # Purpose proportions
    purpose_prop_df.to_excel(writer, sheet_name='Purpose_Proportions', index=False)
    
    # Corridor purpose analysis
    corridor_purpose_df.to_excel(writer, sheet_name='Corridor_Purpose_Analysis', index=False)
    
    # Peak hours by purpose
    peak_purpose_df.to_excel(writer, sheet_name='Peak_Hours_by_Purpose', index=False)
    
    # Hourly data by purpose
    for purpose_name, purpose_label in TRIP_PURPOSES.items():
        hourly_pivot = df_hourly_individual.pivot_table(
            values=purpose_name,
            index=['Middle Filter Zone Name', 'Hour'],
            columns='Day Type',
            aggfunc='sum',
            fill_value=0
        )
        hourly_pivot.to_excel(writer, sheet_name=f'{purpose_name}_Hourly')

In [ ]:
print(f"\nAll results saved to: {excel_path}")
print(f"Heatmaps saved to: {os.path.join(OUTPUT_DIR, 'heatmaps')}")

=============================================================================
KEY FINDINGS
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("KEY FINDINGS SUMMARY")
print("=" * 80)

In [ ]:
# Calculate overall statistics
total_by_purpose = {}
for purpose_name, purpose_label in TRIP_PURPOSES.items():
    total_by_purpose[purpose_label] = df_hourly_individual[purpose_name].sum()

In [ ]:
total_trips = sum(total_by_purpose.values())

In [ ]:
# Calculate outside transit totals
outside_data = df_hourly_individual[
    df_hourly_individual['Transit_Status'].isin(['Before Service', 'After Service'])
]
outside_by_purpose = {}
for purpose_name, purpose_label in TRIP_PURPOSES.items():
    outside_by_purpose[purpose_label] = outside_data[purpose_name].sum()

In [ ]:
print(f"""
{'='*60}
TRIP PURPOSE ANALYSIS SUMMARY
{'='*60}

1. OVERALL TRIP VOLUME BY PURPOSE:
""")

In [ ]:
for purpose, volume in total_by_purpose.items():
    pct = (volume / total_trips * 100) if total_trips > 0 else 0
    print(f"   • {purpose}: {volume:,.0f} ({pct:.1f}%)")

In [ ]:
print(f"""
2. TRIPS OUTSIDE TRANSIT HOURS BY PURPOSE:
""")

In [ ]:
outside_total = sum(outside_by_purpose.values())
for purpose, volume in outside_by_purpose.items():
    pct = (volume / outside_total * 100) if outside_total > 0 else 0
    print(f"   • {purpose}: {volume:,.0f} ({pct:.1f}%)")

In [ ]:
print(f"""
3. PURPOSE COMPOSITION DURING VS OUTSIDE TRANSIT:
""")

In [ ]:
for purpose, volume in total_by_purpose.items():
    outside_vol = outside_by_purpose.get(purpose, 0)
    outside_pct = (outside_vol / volume * 100) if volume > 0 else 0
    during_vol = volume - outside_vol
    during_pct = 100 - outside_pct
    print(f"   • {purpose}:")
    print(f"     - During transit: {during_vol:,.0f} ({during_pct:.1f}%)")
    print(f"     - Outside transit: {outside_vol:,.0f} ({outside_pct:.1f}%)")

In [ ]:
print(f"""
4. CORRIDORS WITH HIGHEST OUTSIDE-TRANSIT DEMAND BY PURPOSE:
""")

In [ ]:
for purpose_name, purpose_label in TRIP_PURPOSES.items():
    corridor_totals = []
    for corridor in COMMERCIAL_CORRIDORS:
        corridor_data = df_hourly_individual[
            df_hourly_individual['Middle Filter Zone Name'] == corridor
        ]
        outside_corridor = corridor_data[
            corridor_data['Transit_Status'].isin(['Before Service', 'After Service'])
        ][purpose_name].sum()
        corridor_totals.append((corridor, outside_corridor))
    
    corridor_totals.sort(key=lambda x: x[1], reverse=True)
    print(f"\n   {purpose_label}:")
    for corridor, volume in corridor_totals[:3]:
        if volume > 0:
            print(f"     • {corridor}: {volume:,.0f} trips outside transit")

In [ ]:
print(f"""
5. RECOMMENDATIONS:

Based on the trip purpose analysis:
""")

In [ ]:
# Generate recommendations based on findings
hbw_outside_pct = (outside_by_purpose.get('Home-Based Work', 0) / 
                   total_by_purpose.get('Home-Based Work', 1) * 100)
hbo_outside_pct = (outside_by_purpose.get('Home-Based Other', 0) / 
                   total_by_purpose.get('Home-Based Other', 1) * 100)
nhb_outside_pct = (outside_by_purpose.get('Non-Home Based', 0) / 
                   total_by_purpose.get('Non-Home Based', 1) * 100)

In [ ]:
if hbw_outside_pct > 20:
    print(f"   • WORK TRIPS: {hbw_outside_pct:.1f}% of work trips occur outside transit hours")
    print(f"     Consider early morning/late evening service for commuters")

In [ ]:
if hbo_outside_pct > 25:
    print(f"   • OTHER HOME-BASED TRIPS: {hbo_outside_pct:.1f}% occur outside transit hours")
    print(f"     This suggests demand for shopping/errand travel outside service hours")

In [ ]:
if nhb_outside_pct > 30:
    print(f"   • NON-HOME BASED TRIPS: {nhb_outside_pct:.1f}% occur outside transit hours")
    print(f"     These are often multi-purpose trips that may need flexible service")

In [ ]:
print(f"""
6. NEXT STEPS:
   • Review purpose-specific heatmaps in the 'heatmaps' folder
   • Check complete_trip_purpose_analysis.xlsx for detailed breakdowns
   • Compare purpose patterns across different corridors
   • Consider targeted surveys for purposes with high outside-transit demand
""")

In [ ]:
print("\n" + "=" * 80)
print(f"ANALYSIS COMPLETE: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)